# Property Value Preprocessing

In [1]:
import numpy as np
import pandas as pd
import janitor
import geopandas as gpd

In [13]:
census_year = 2020

if census_year == 2020:
    value = pd.read_csv("data/2020_property_value/ACSDT5Y2020.B25077-Data.csv")
    val_col = 'B25077_001M'
    na_filler = "**"
elif census_year == 2010:
    value = pd.read_csv("data/2010_property_value/ACSDT5Y2010.B25077-Data.csv")
    val_col = 'B25077_001E'
    na_filler = "-"

value.head()

,GEO_ID,NAME,B25077_001E,B25077_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Median value (dollars),Margin of Error!!Median value (dollars),NaN
1,1400000US42101000101,"Census Tract 1.01, Philadelphia County, Pennsy...",414000,93462,NaN
2,1400000US42101000102,"Census Tract 1.02, Philadelphia County, Pennsy...",456900,98400,NaN
3,1400000US42101000200,"Census Tract 2, Philadelphia County, Pennsylvania",448100,190300,NaN
4,1400000US42101000300,"Census Tract 3, Philadelphia County, Pennsylvania",627900,75611,NaN


In [15]:
# check total census tracts with NA property value
print(np.sum(value[val_col] == na_filler))
value[val_col] = np.where(value[val_col] == na_filler, None, value[val_col])
value[val_col]

29


0      Margin of Error!!Median value (dollars)
1                                        93462
2                                        98400
3                                       190300
4                                        75611
                        ...                   
404                                       None
405                                       None
406                                       None
407                                       None
408                                       None
Name: B25077_001M, Length: 409, dtype: object

In [16]:
# select the overall median estimate S1903_C02_001E
med_prop_val = value[['GEO_ID', 'NAME', val_col]]
med_prop_val = med_prop_val.iloc[1:]

med_prop_val['tract'] = med_prop_val['NAME'].str.extract(r"Census Tract ([0-9.]+)")
med_prop_val['tract'] = med_prop_val['tract'].astype(float).astype(str)
med_prop_val.drop(columns=['NAME'], inplace=True)
med_prop_val.rename(columns={val_col: "med_prop_val"}, inplace=True)
med_prop_val.head()

,GEO_ID,med_prop_val,tract
1,1400000US42101000101,93462,1.01
2,1400000US42101000102,98400,1.02
3,1400000US42101000200,190300,2.0
4,1400000US42101000300,75611,3.0
5,1400000US42101000401,145592,4.01


In [17]:
# check for census tracts do not have an estimated median property value?
med_prop_val.shape[0]

408

In [18]:
med_prop_val.to_csv(f"data/med_prop_val_{census_year}.csv")